In [481]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''
Created on 2024-05-27
@author: Juan Enrique López
@description: Jupyter Notebook creado para descargar las reglas de diferentes fuentes y clasificarlas por TTP
'''

'\nCreated on 2024-05-27\n@author: Juan Enrique López\n@description: Jupyter Notebook creado para descargar las reglas de diferentes fuentes y clasificarlas por TTP\n'

**Importante**: UST-Ciberseguridad nos pide no subir los outputs a sharepoint

In [482]:
import os
#from os import remove
import re
import shutil
import wget

import zipfile
import yaml
import csv
import toml
import yara

from pathlib import Path
from collections import Counter

from stix2 import Filter, MemoryStore
import stix2
import requests

import pandas as pd

#### **Funciones**

In [483]:
def create_output_folder(path):
    '''
    Función creada para crear el directorio facilitado en caso de no existir previamente.
    '''
    if not os.path.exists(path):
        os.makedirs(path)

In [484]:
def find_techniques(texto, techniques_list):
    '''
    Función que, dado un str, comprueba  si existe alguna técnica en el texto facilitado.
    En caso de encontrarse alguna técnica se añade a una lista que será devuelta al final. En caso de no encontrarse devuelve un vacío.
    '''
    if isinstance(texto, str):
        found = []
        for item in techniques_list:
            if item in texto:
                found.append(item)
        return ', '.join(found)
    else:
        return ''

In [485]:
def find_techniques_in_list(cadena, techniques_list):
    '''
    Función que, dado un str, comprueba en la lista de técnicas facilitada si existe dicha técnica y devuelve la cadena (ttp). En caso de no encontrarse devuelve None.
    '''
    for item in techniques_list:
        if item in cadena:
            return item
    return None

In [486]:
def get_unique_ttps_generated(paths):
    sub_folders = set()
    for path in paths:
        dir, file = os.path.split(path)
        last_sub_folder = os.path.basename(dir)
        sub_folders.add(last_sub_folder)
    return list(sub_folders) 

In [487]:
def get_dict_ttps_rules_asigned(paths):
    from collections import defaultdict
    file_count = defaultdict(int)
    for path in paths:
        last_subfolder = os.path.basename(os.path.dirname(path))
        file_count[last_subfolder] += 1
    resume = dict(file_count)
    resume = dict(sorted(resume.items(), key=lambda item: item[1], reverse=True))
    return resume

In [488]:
def sentineldf_to_md(df, path_save, techniques_list):
    '''
    Función creada para guardar las reglas de deteccion Sentinel obtenidas a partir del output de get_ttps_from_CP_files
    '''
    num_rules = []
    for index, row in df.iterrows():
        if find_techniques_in_list(row['technique_id'], techniques_list) != None: # Esta comprobación es técnicamente innecesaria ya que las ttp ya vienen clasificadas por matriz, por lo que ninguna deberia ir a T0000
            ttp = row['technique_id']
        else: 
            ttp = 'T0000'
        chars_to_replace = r'[\/\\\"\<\>\|\*\:]'
        name = re.sub(chars_to_replace, ' ', row['title'])
        filename = os.path.join(path_save, ttp, f"{row['technique_id']} - {name[:50]}.md") # El nombre del fichero solo estará compuesto por los primeros 50 caracteres
        filename.replace('/', '')
        
        create_output_folder(os.path.join(path_save, ttp))

        with open(filename, 'w') as file:
            file.write(f"**technique_id**\n{row['technique_id']}\n\n")
            file.write(f"**title**\n{row['title']}\n\n")
            file.write(f"**query**\n{row['query']}\n\n")
            file.write(f"**source**\n{row['source']}\n\n")
            file.write(f"**type_source**\n{row['type_source']}\n\n")
            print(f"Archivo guardado correctamente: {filename}")
            num_rules.append(filename)
    print('Proceso finalizado!')
    return num_rules

In [489]:
def splunkdf_to_md(df, path_save, techniques_list):
    '''
    Función creada para guardar las reglas de deteccion Splunk obtenidas a partir del output de get_ttps_from_CP_files
    '''
    num_rules = []
    for index, row in df.iterrows():
        if find_techniques_in_list(row['technique_id'], techniques_list) != None: # Esta comprobación es técnicamente innecesaria ya que las ttp ya vienen clasificadas por matriz, por lo que ninguna deberia ir a T0000
            ttp = row['technique_id']
        else: 
            ttp = 'T0000'
        chars_to_replace = r'[\/\\\"\<\>\|\*\:]'
        name = re.sub(chars_to_replace, ' ', row['title'])
        filename = os.path.join(path_save, ttp, f"{row['technique_id']} - {name[:50]}.md") # El nombre del fichero solo estará compuesto por los primeros 50 caracteres
        filename.replace('/', '')
        
        create_output_folder(os.path.join(path_save, ttp))

        with open(filename, 'w') as file:
            file.write(f"**technique_id**\n{row['technique_id']}\n\n")
            file.write(f"**title**\n{row['title']}\n\n")
            file.write(f"**description**\n{row['description']}\n\n")
            file.write(f"**query**\n{row['query']}\n\n")
            file.write(f"**source**\n{row['source']}\n\n")
            file.write(f"**type_source**\n{row['type_source']}\n\n")
            print(f"Archivo guardado correctamente: {filename}")
            num_rules.append(filename)
    print('Proceso finalizado!')
    return num_rules

In [490]:
def qradardf_to_md(df, path_save, techniques_list):
    '''
    Función creada para guardar las reglas de deteccion Qradar obtenidas a partir del output de get_ttps_from_CP_files
    '''
    num_rules = []
    for index, row in df.iterrows():
        if find_techniques_in_list(row['technique_id'], techniques_list) != None: # Esta comprobación es técnicamente innecesaria ya que las ttp ya vienen clasificadas por matriz, por lo que ninguna deberia ir a T0000
            ttp = row['technique_id']
        else: 
            ttp = 'T0000'
        chars_to_replace = r'[\/\\\"\<\>\|\*\:]'
        name = re.sub(chars_to_replace, ' ', row['rule'])
        filename = os.path.join(path_save, ttp, f"{row['technique_id']} - {name[:50]}.md") # El nombre del fichero solo estará compuesto por los primeros 50 caracteres
        filename.replace('/', '')
        
        create_output_folder(os.path.join(path_save, ttp))

        with open(filename, 'w') as file:
            file.write(f"**technique_id**\n{row['technique_id']}\n\n")
            file.write(f"**rule**\n{row['title']}\n\n")
            file.write(f"**source**\n{row['source']}\n\n")
            file.write(f"**type_source**\n{row['type_source']}\n\n")
            print(f"Archivo guardado correctamente: {filename}")
            num_rules.append(filename)
    print('Proceso finalizado!')
    return num_rules

In [491]:
def download_unzip(url_zip, temp_path):
    #Descarga el fichero pasado por parametro
    wget.download(url_zip, os.path.join(temp_path, 'temp.zip'))
    #Descomprimir el fichero descargado y se borra el fichero descargado
    f_zip = zipfile.ZipFile(os.path.join(temp_path, 'temp.zip'))
    try:
        f_zip.extractall(path=temp_path)
    except:
        print('ERROR: MAX_PATH longer than 256 characters')
    f_zip.close()
    os.remove(os.path.join(temp_path, 'temp.zip'))

In [492]:
def download_folder_unzip(ruta, folder, temp_path):
    try:
        archivo_zip = wget.download(ruta, os.path.join(temp_path, 'temp.zip'))
        with zipfile.ZipFile(archivo_zip, 'r') as f_zip:
            for archivo in f_zip.namelist():
                if archivo.startswith(folder):
                    f_zip.extract(archivo, path=temp_path)
        os.remove(archivo_zip)
        print("Descarga y descompresión completadas correctamente.")
    except Exception as e:
        print('Error:', e)

In [493]:
def get_list_of_files_sub(dirName):
    # Crea lista de ficheros y subdirectorios
    listOfFile = os.listdir(dirName)
    allFiles = list()
    #Recorre los directorios y subdirectorios y genera una lista con los ficheros encontrados en estos
    for entry in listOfFile:
        fullPath = os.path.join(dirName, entry)
        if os.path.isdir(fullPath):
            allFiles = allFiles + get_list_of_files_sub(fullPath)
        else:
            allFiles.append(fullPath)
    #Devuelve la lista con los ficheros
    return allFiles

In [494]:
def get_folders(path):
    if not os.path.exists(path):
        raise ValueError("La ruta proporcionada no existe.")
    if not os.path.isdir(path):
        raise ValueError("La ruta proporcionada no es un directorio.")
    folders = [name for name in os.listdir(path) if os.path.isdir(os.path.join(path, name))]
    return folders

In [495]:
def delete_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Directorio {path} eliminado correctamente.")
    except PermissionError as e:
        print(f"Error de permisos al intentar eliminar el directorio: {e}")
    except FileNotFoundError as e:
        print(f"El directorio no existe: {e}")
    except Exception as e:
        print(f"Ocurrió un error al eliminar el directorio: {e}")

In [496]:
# Deprecated
# def techniques():
#     lift = attack_client()
#     #Solicitud a la libreria attackcti la extraccion de las tecnicas Enterprise y normalizar el json en un dataframe
#     techniques = lift.get_enterprise_techniques(stix_format=False)
#     techniques = pd.json_normalize(techniques)
#     #Eliminar las tecnicas deprecadas y revocadas
#     techniques = techniques[(techniques['mitre_deprecated'] != True)]
#     # Eliminamos duplicados y convertimos en lista
#     techniques = techniques['technique_id'].drop_duplicates().tolist()
#     techniques = sorted(techniques, key=len, reverse=True)
#     return techniques

In [497]:
def get_data_from_branch(matrix):
    url = f"https://raw.githubusercontent.com/mitre/cti/master/{matrix}-attack/{matrix}-attack.json"
    stix_json = requests.get(url).json()
    return MemoryStore(stix_data=stix_json["objects"])

In [498]:
# Obtención de la lista de técnicas de los datos facilitados. Por defecto se retornan técnicas y subtécnicas pudiendose seleccionar
def get_list_techniques_from_stix2(src, include="both"):
    if include == "techniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', False)
        ])
    elif include == "subtechniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', True)
        ])
    elif include == "both":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern')
        ])
    else:
        raise RuntimeError("Unknown option %s!" % include)
    
    if isinstance(query_results, stix2.datastore.memory.MemoryStore):
        query_results = query_results.query()
        stix2_df = pd.DataFrame(query_results)
    elif isinstance(query_results[0], stix2.v20.sdo.AttackPattern):
        stix2_df = pd.DataFrame(query_results)

    stix2_df['technique_id'] = stix2_df['external_references'].apply(lambda refs: refs[0].external_id if refs else None)
    stix2_df = stix2_df[(stix2_df['revoked']!=True)&(stix2_df['x_mitre_deprecated']!=True)]
    stix2_df['technique_id'] = stix2_df['technique_id'].str.upper()
    techniques = sorted(stix2_df['technique_id'].drop_duplicates(), key=len, reverse=True)

    return techniques

In [499]:
def clean_names(name):
    delete_chars = r'[\/\\|,";:#\[\]]'
    name = re.sub(delete_chars, '', name)
    name = " ".join(name.split())
    return name.strip()

In [500]:
def copy_file_to_path(item_path, save_path, ttp_name):
    if not os.path.exists(os.path.join(save_path, ttp_name)):
        os.makedirs(os.path.join(save_path, ttp_name))
    shutil.copy(item_path,os.path.join(save_path, ttp_name))
    print('Regla asignada correctamente a la carpeta de la TTP: '+ ttp_name) 

In [501]:
class NotTTPatTags(Exception):
    def __init__(self, mensaje):
        self.mensaje = mensaje
        super().__init__(self.mensaje)

In [502]:
def get_unique_items(list_paths):
    from pathlib import Path
    unique_list = []
    for item_path in list_paths:
        unique_list.append(Path(item_path).name)
    unique_list = list(set(unique_list))
    return unique_list


In [503]:
def copy_files_between_paths(path_source, path_destiny):
    if not os.path.exists(path_source):
        raise ValueError(f"El directorio de origen '{path_source}' no existe.")
    if not os.path.isdir(path_source):
        raise ValueError(f"'{path_source}' no es un directorio.")
    # Si el directorio de destino no existe, crearlo
    if not os.path.exists(path_destiny):
        os.makedirs(path_destiny)
    for item in os.listdir(path_source):
        if item != '.DS_Store': 
            src_path = os.path.join(path_source, item)
            dst_path = os.path.join(path_destiny, item)
            if os.path.isdir(src_path):
                shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
            else:
                shutil.copy2(src_path, dst_path)

In [504]:
def sigma7_classifier_techniques(path_source, path_destiny, techniques_list):
    '''
    Función especial para Sigma HQ 7. En este caso los ficheros descargados vienen clasificados por TTP, pero antes de copiar a outputs debemos de filtrar por extensión del archivo ya que vienen .png adicionales
    y copiaremos solamente las reglas relativas a la matriz analizada, moviendo el resto de reglas a la carpeta T0000.
    '''
    num_rules = []
    if not os.path.exists(path_source):
        raise ValueError(f"El directorio de origen '{path_source}' no existe.")
    if not os.path.isdir(path_source):
        raise ValueError(f"'{path_source}' no es un directorio.")
    
    # Si el directorio de destino no existe, crearlo
    if not os.path.exists(path_destiny):
        os.makedirs(path_destiny)
    
    t0000_path = os.path.join(path_destiny, 'T0000')
    # Extensiones permitidas: para evitar duplicidades nos quedamos únicamente con .md / .markdown
    # allowed_extensions = {'.md', '.markdown', '.yml', '.yaml'}
    allowed_extensions = {'.md', '.markdown'}
    for item in os.listdir(path_source):
        if item != '.DS_Store':
            src_path = os.path.join(path_source, item)
            if item in techniques_list:
                dst_path = os.path.join(path_destiny, item)
                if os.path.isdir(src_path):
                    if not os.path.exists(dst_path):
                        os.makedirs(dst_path)
                    for file_name in os.listdir(src_path):
                        file_src_path = os.path.join(src_path, file_name)
                        if os.path.isfile(file_src_path) and os.path.splitext(file_name)[1] in allowed_extensions:
                            file_dst_path = os.path.join(dst_path, file_name)
                            shutil.copy2(file_src_path, file_dst_path)
                            num_rules.append(file_src_path)
                else:
                    if os.path.splitext(item)[1] in allowed_extensions:
                        shutil.copy2(src_path, dst_path)
                        num_rules.append(src_path)
            else:
                if not os.path.exists(t0000_path):
                    os.makedirs(t0000_path)
                if os.path.isdir(src_path):
                    for file_name in os.listdir(src_path):
                        file_src_path = os.path.join(src_path, file_name)
                        if os.path.isfile(file_src_path) and os.path.splitext(file_name)[1] in allowed_extensions:
                            file_dst_path = os.path.join(t0000_path, file_name)
                            shutil.copy2(file_src_path, file_dst_path)
                            num_rules.append(file_src_path)
                else:
                    if os.path.splitext(item)[1] in allowed_extensions:
                        shutil.copy2(src_path, os.path.join(t0000_path, item))
                        num_rules.append(src_path)
    print('Proceso finalizado!')

    return num_rules

In [505]:
#Función para devolver el numero de reglas que han sido asignadas a 2 o más ttps
def count_duplicate_rules(paths):
    # Extraer nombres de archivo de las rutas
    filenames = [os.path.basename(path) for path in paths]
    # Contar archivos repetidos
    file_counts = Counter(filenames)
    # Obtener solo los archivos que están repetidos
    duplicates = {filename: count for filename, count in file_counts.items() if count > 1}
    return duplicates

In [506]:
def yaml_classifier_techniques(files_list, techniques_list, save_path):
    '''
    Función encargada de buscar el contenido en los ficheros .yaml.
    '''
    num_rules = []
    ttp_in_filename = []
    content_in_yaml = []
    no_content_in_yaml = []
    ttp_in_yaml = []
    error_load = []

    for item in files_list:
        root, extension = os.path.splitext(item)
        if extension == '.yaml' or extension == '.yml':
            with open(item, 'r', encoding='utf-8') as file:
                try:
                    # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y se revisa el nombre del fichero antes de copiar el fichero a la carpeta T0000
                    yaml_content = file.read()
                    yaml_content = yaml_content.upper()
                    num_rules.append(item)
                    # Revisamos que se haya podido obtener texto del contenido del .yaml 
                    if isinstance(yaml_content, str):
                        ttps = find_techniques(yaml_content, techniques_list)
                        ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                        content_in_yaml.append(item)
                    else:
                        #Si el contenido no dispone de texto, buscamos ttps en el propio nombre del fichero
                        ttps = find_techniques(item.upper(), techniques_list)
                        ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                        no_content_in_yaml.append(item)
                    if len(ttps)>0:
                        # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                        for ttp in ttps:
                            if ttp != '':
                                ttp_in_yaml.append(item)
                                copy_file_to_path(item, save_path, ttp)
                    
                    #En cualquier caso, siempre buscamos ttp en el nombre del fichero
                    ttps = find_techniques(item.upper(), techniques_list)
                    ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                    if len(ttps)>0:
                        # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                        for ttp in ttps:
                            if ttp != '':
                                ttp_in_filename.append(item)
                                copy_file_to_path(item, save_path, ttp)
                    else:
                        copy_file_to_path(item, save_path, 'T0000')

                # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
                except Exception as e:
                    print(f"Error al intentar leer el archivo yaml: {e}")
                    error_load.append(item)
                    ttps = find_techniques(item.upper(), techniques_list)
                    if len(ttps)>0:
                        # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                        for ttp in ttps:
                            if ttp != '':
                                ttp_in_filename.append(item)
                                copy_file_to_path(item, save_path, ttp)
                    else:
                        copy_file_to_path(item, save_path, 'T0000')
    
    # return num_rules, ttp_in_filename, content_in_yaml, no_content_in_yaml, ttp_in_yaml, error_load
    return num_rules

In [507]:
def txt_basename_classifier_techniques(files_list, techniques_list, save_path):
    '''
    Función encargada de clasificar los archivos txt por ttp encontrada en la lista facilitada. La búsqueda se restringe al nombre del fichero dadas las caracteristicas de Netevert.
    '''
    num_rules = []
    ttp_basename_found = []
    ttp_basename_not_found = []

    for item in files_list:
        root,extension = os.path.splitext(item)
        if extension == '.txt':
            num_rules.append(item)
            basename = os.path.basename(item)
            basename_ttp = basename[:5]
            ttp_found = find_techniques_in_list(basename_ttp, techniques_list)
            if not ttp_found:
                ttp_folder = 'T0000'
                ttp_basename_not_found.append(item)
                print(f'No identificada ttp {basename_ttp} en el archivo {item}')
            else:
                ttp_folder = ttp_found
                ttp_basename_found.append(item)
                print(f'Identificada ttp {ttp_found} en el archivo {item}')
            create_output_folder(os.path.join(save_path, ttp_folder))
            shutil.copy(item, os.path.join(save_path, ttp_folder))

    print(f'Proceso finalizado!')
    # return num_rules, ttp_basename_found, ttp_basename_not_found
    return num_rules

In [508]:
def toml_find_techniques(toml_dict, techniques_list):
    '''
    Función encargada de encontrar técnicas dentro de un archivo .toml leido como diccionario. Buscará en la profundidad completa del diccionario. 
    '''
    found_values = []
    if isinstance(toml_dict, dict):
        for key, value in toml_dict.items():
            if isinstance(value, dict):
                found_values.extend(toml_find_techniques(value, techniques_list))
            elif isinstance(value, list):
                for item in value:
                    if isinstance(item, dict):
                        found_values.extend(toml_find_techniques(item, techniques_list))
                    elif item in techniques_list:
                        found_values.append(item)
            elif value in techniques_list:
                found_values.append(value)
                
    return found_values

In [509]:
def toml_classifier_techniques_elastic(files_list, techniques_list, save_path):
    '''
    Función encargada de clasificar el listado de archivos .toml por ttp encontrada en el propio fichero. Se llama a la función toml_find_techniques() que permite buscar en la estructura completa
    del diccionario generado por toml.load(file).
    '''
    num_rules = []
    load_toml = []
    error_toml = []
    ttp_found = []
    ttp_not_found = []
    ttp_identify = []
    
    for item in files_list:
        root,extension = os.path.splitext(item)
        if extension == '.toml':
            num_rules.append(item)
            with open(item, 'r') as file:
                try:
                    data_toml = toml.load(file)
                    load_toml.append(item)
                except (IndexError, toml.TomlDecodeError) as e:
                    error_toml.append(item)
                    print(f'{e} - No se ha podido asignar ninguna carpeta al archivo: {item}')
                found_techniques = toml_find_techniques(data_toml, techniques_list)
                if len(found_techniques)>0:
                    ttp_found.append(item)
                    print(f"Identificada/s ttp's {found_techniques} en: {item}")
                    for ttp in found_techniques:
                        ttp_identify.append(ttp)
                        create_output_folder(os.path.join(save_path, ttp))
                        # if not os.path.exists(os.path.join(save_path, ttp)):
                        #     os.makedirs(os.path.join(save_path, ttp))
                        shutil.copy(item,os.path.join(save_path, ttp))
                else:
                    ttp_not_found.append(item)
                    print(f"No se ha podido identificar ninguna ttp en: {item}")
                    create_output_folder(os.path.join(save_path, 'T0000'))
                    shutil.copy(item, os.path.join(save_path, 'T0000'))

    # return num_toml, load_toml, error_toml, ttp_found, ttp_not_found, ttp_identify
    return num_rules

In [510]:
def md_classifier_techniques(files_list, techniques_list, save_path):
    '''
    Función encargada de clasificar los archivos .md o .markdown por ttp encontrada en la lista facilitada. La búsqueda se restringe al nombre del fichero dadas las caracteristicas de Netevert.
    '''
    num_rules = []
    ttp_in_filename = []
    content_in_md = []
    no_content_in_md = []
    ttp_in_md = []
    error_load = []

    for item in files_list:
        root, extension = os.path.splitext(item)
        if extension == '.md' or extension == '.markdown':
            with open(item, 'r', encoding='utf-8') as file:
                try:
                    # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y se revisa el nombre del fichero antes de copiar el fichero a la carpeta T0000
                    md_content = file.read()
                    md_content = md_content.upper()
                    num_rules.append(item)
                    # Revisamos que se haya podido obtener texto del contenido del .md 
                    if isinstance(md_content, str):
                        ttps = find_techniques(md_content, techniques_list)
                        ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                        content_in_md.append(item)
                    else:
                        #Si el contenido no dispone de texto, buscamos ttps en el propio nombre del fichero
                        ttps = find_techniques(item.upper(), techniques_list)
                        ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                        no_content_in_md.append(item)
                    if len(ttps)>0:
                        # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                        for ttp in ttps:
                            if ttp != '':
                                ttp_in_md.append(item)
                                copy_file_to_path(item, save_path, ttp)
                    
                    #En cualquier caso, siempre buscamos ttp en el nombre del fichero
                    ttps = find_techniques(item.upper(), techniques_list)
                    ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                    if len(ttps)>0:
                        # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                        for ttp in ttps:
                            if ttp != '':
                                ttp_in_filename.append(item)
                                copy_file_to_path(item, save_path, ttp)
                    else:
                        copy_file_to_path(item, save_path, 'T0000')

                # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
                except Exception as e:
                    print(f"Error al intentar leer el archivo markdown: {e}")
                    error_load.append(item)
                    ttps = find_techniques(item.upper(), techniques_list)
                    if len(ttps)>0:
                        # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                        for ttp in ttps:
                            if ttp != '':
                                ttp_in_md.append(item)
                                copy_file_to_path(item, save_path, ttp)
                    else:
                        copy_file_to_path(item, save_path, 'T0000')

    # return num_mds, ttp_in_filename, content_in_md, no_content_in_md, ttp_in_md, error_load
    return num_rules

In [511]:
def yara_classifier_techniques(files_list, techniques_list, save_path):
    num_rules = []
    ttp_in_filename = []
    content_in_yara = []
    no_content_in_yara = []
    ttp_in_yara = []
    error_load = []

    for item in files_list:
        root, extension = os.path.splitext(item)
        if extension == '.yar' or extension == '.yara':
            with open(item, 'r', encoding='utf-8') as file:
                try:
                    # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y se revisa el nombre del fichero antes de copiar el fichero a la carpeta T0000
                    md_content = file.read()
                    md_content = md_content.upper()
                    num_rules.append(item)
                    # Revisamos que se haya podido obtener texto del contenido del .yar 
                    if isinstance(md_content, str):
                        ttps = find_techniques(md_content, techniques_list)
                        ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                        content_in_yara.append(item)
                    else:
                        #Si el contenido no dispone de texto, buscamos ttps en el propio nombre del fichero
                        ttps = find_techniques(item.upper(), techniques_list)
                        ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                        no_content_in_yara.append(item)
                    if len(ttps)>0:
                        # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                        for ttp in ttps:
                            if ttp != '':
                                ttp_in_yara.append(item)
                                copy_file_to_path(item, save_path, ttp)
                    
                    #En cualquier caso, siempre buscamos ttp en el nombre del fichero
                    ttps = find_techniques(item.upper(), techniques_list)
                    ttps = [item.strip() for item in ttps.split(',') if item.strip()]
                    if len(ttps)>0:
                        # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                        for ttp in ttps:
                            if ttp != '':
                                ttp_in_filename.append(item)
                                copy_file_to_path(item, save_path, ttp)
                    else:
                        copy_file_to_path(item, save_path, 'T0000')

                # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
                except Exception as e:
                    print(f"Error al intentar leer el archivo yara: {e}")
                    error_load.append(item)
                    ttps = find_techniques(item.upper(), techniques_list)
                    if len(ttps)>0:
                        # Copiamos cada una de las ttps encontradas en su carpeta correspondiente
                        for ttp in ttps:
                            if ttp != '':
                                ttp_in_filename.append(item)
                                copy_file_to_path(item, save_path, ttp)
                    else:
                        copy_file_to_path(item, save_path, 'T0000')

    # return num_yaras, ttp_in_filename, content_in_yara, no_content_in_yara, ttp_in_yara, error_load
    return num_rules

In [512]:
# Deprecated
# Adaptación de la anterior función para el caso de las fuentes Sigma. En este caso construimos un arbol de búsquedas específico para revisar las propiedades de los ficheros de esta fuente.
# def old_yaml_classifier_techniques_sigma(files_list, techniques_list, save_path):
#     # Recorrer la lista de archivos
#     num_rules = []
#     has_label_tags = []
#     has_label_with_content_tags = []
#     has_label_without_content_tags = []
#     no_label_tags = []
#     not_ttp_in_tags = []
#     not_ttp_in_tags_with_ttp_in_description = []
#     not_ttp_in_tags_without_description = []
#     error_load = []

#     for item in files_list:
#         root, extension = os.path.splitext(item)
#         if extension == '.yaml' or extension == '.yml':
#             with open(item) as file:
#                 num_rules.append(item)
#                 try:
#                     # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y copiamos el fichero a la carpeta T0000
#                     document = yaml.full_load(file)
#                     if 'tags' in document:
#                         # Si encontramos la etiqueta 'tags' en el yaml lo guardamos como una variable 
#                         tags = document.get('tags', [])
#                         has_label_tags.append(item)
#                         # Si el resultado de esa variable no está vacío y contiene información lo convertimos en una lista
#                         if not isinstance(tags, type(None)):
#                             has_label_with_content_tags.append(item)
#                             # Convertimos en lista el contenido de la etiqueta
#                             #tags = [tags]
#                             tags = [tag.upper() for tag in tags if isinstance(tag, str)]
#                             # Recorremos la lista de items contenidos en tags y contamos las ttp encontradas ya que si no encontramos ninguna, pese haber texto en la etiqueta
#                             # deberemos mover el fichero a la carpeta T0000
#                             ttp_counter = 0
#                             for tag in range(len(tags)):
#                                 # Matcheamos el item de la lista tags con nuestra lista de ttps 
#                                 ttp = find_techniques_in_list(tags[tag],techniques_list)
#                                 # Y si es un string lo que tenemos, entonces guardamos en la carpeta de la la ttp correspondiente
#                                 # Ojo,¿que pasa si la lista tiene un string pero no una ttp?
#                                 if isinstance(ttp, str):
#                                     ttp_counter += 1
#                                     copy_file_to_path(item, save_path, ttp)
#                             if ttp_counter == 0:
#                                 raise NotTTPatTags('.yaml sin TTP encontrada en los tags')
                    
#                         else:
#                             # En el caso de que la etiquete exista pero su contenido sea vacio (NoneType), accedemos a la etiqueta 'description' para buscar las TTP´s
#                             has_label_without_content_tags.append(item)
#                             if 'description' in document:
#                                 description = [document.get('description', [])]
#                                 # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
#                                 for tag in range(len(description)):
#                                     ttp = find_techniques_in_list(description[tag],techniques_list)
#                                     if not isinstance(ttp, type(None)):
#                                         if not os.path.exists(os.path.join(save_path, ttp)):
#                                             os.makedirs(os.path.join(save_path, ttp))
#                                     else:
#                                         ttp='T0000'
#                                     copy_file_to_path(item, save_path, ttp)
#                     else:
#                         no_label_tags.append(item)
#                         # Repetimos la busqueda de TTP en la etiqueta description en los casos en los que 'tags' no exista.
#                         if 'description' in document:
#                             description = [document.get('description', [])]
#                             # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
#                             for tag in range(len(description)):
#                                 ttp = find_techniques_in_list(description[tag],techniques_list)
#                                 if not isinstance(ttp, type(None)):
#                                     if not os.path.exists(os.path.join(save_path, ttp)):
#                                         os.makedirs(os.path.join(save_path, ttp))
#                                 else:
#                                     ttp='T0000'
#                                 copy_file_to_path(item, save_path, ttp)
#                         else:
#                             copy_file_to_path(item, save_path, 'T0000')
#                 # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
#                 except yaml.YAMLError as e:
#                     print(f"Error al cargar el archivo YAML: {e}")
#                     error_load.append(item)
#                     copy_file_to_path(item, save_path, 'T0000')
                
#                 # Con esta excepción llegamos si hemos recorrido la lista de tags y no hemos encontrado ninguna TTP, por lo que revisamos la etiqueta dando lugar a 3 casuísticas,
#                 # la primera que encuentre dicha etiqueta, en cuyo caso se revisa el contenido, si dentro de este se encuentra alguna TTP se mandara a la carpeta correspondiente. En caso de no haber coincidencia, al igual que ocurrirá si no hay etiqueta 'description' el fichero será copiado a T000.
#                 except NotTTPatTags as e:
#                     print(f"{e}")
#                     not_ttp_in_tags.append(item)
#                     if 'description' in document:
#                         description = [document.get('description', [])]
#                         # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
#                         for tag in range(len(description)):
#                             ttp = find_techniques_in_list(description[tag],techniques_list)
#                             if not isinstance(ttp, type(None)):
#                                 not_ttp_in_tags_with_ttp_in_description.append(item)
#                                 pass
#                             else:
#                                 ttp='T0000'
#                             copy_file_to_path(item, save_path, ttp)
#                     else:
#                         not_ttp_in_tags_without_description.append(item)
#                         copy_file_to_path(item, save_path, 'T0000')
#     # return num_rules, has_label_tags, has_label_with_content_tags, has_label_without_content_tags, no_label_tags, not_ttp_in_tags, not_ttp_in_tags_with_ttp_in_description, not_ttp_in_tags_without_description, error_load
#     return num_rules

In [513]:
def yaml_classifier_techniques_sigma(files_list, techniques_list, save_path):
    num_rules = []
    has_label_tags = []
    has_label_with_content_tags = []
    has_label_without_content_tags = []
    no_label_tags = []
    not_ttp_in_tags = []
    not_ttp_in_tags_with_ttp_in_description = []
    not_ttp_in_tags_without_description = []
    error_load = []
    error_load_ttp_in_path_name = []

    ttps_in_tags_not_in_techniques = []
    ttps_in_tags_not_in_techniques_items = []

    for item in files_list:
        root, extension = os.path.splitext(item)
        if extension == '.yaml' or extension == '.yml':
            with open(item) as file:
                num_rules.append(item)
                try:
                    # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y copiamos el fichero a la carpeta T0000
                    document = yaml.full_load(file)
                    if 'tags' in document:
                        # Si encontramos la etiqueta 'tags' en el yaml lo guardamos como una variable 
                        tags = document.get('tags', [])
                        has_label_tags.append(item)
                        # Si el resultado de esa variable no está vacío y contiene información lo convertimos en una lista
                        if not isinstance(tags, type(None)):
                            has_label_with_content_tags.append(item)
                            # Convertimos en lista el contenido de la etiqueta
                            #tags = [tags]
                            tags = [tag.upper() for tag in tags if isinstance(tag, str)]
                            # Recorremos la lista de items contenidos en tags y contamos las ttp encontradas ya que si no encontramos ninguna, pese haber texto en la etiqueta
                            # deberemos mover el fichero a la carpeta T0000
                            ttp_counter = 0
                            for tag in range(len(tags)):
                                # Matcheamos el item de la lista tags con nuestra lista de ttps 
                                ttp = find_techniques_in_list(tags[tag], techniques_list)
                                # He encontrado en sigma HQ 5 tags que contienen técnicas que no están en el listado de techniques_enterprise, vamos a almacenarlas para reportarlas.
                                if '.T0' in tags[tag] or '.T1' in tags[tag]:
                                    ttps_in_tags_not_in_techniques.append(tags[tag])
                                    ttps_in_tags_not_in_techniques_items.append(item)
                                # Y si es un string lo que tenemos, entonces guardamos en la carpeta de la la ttp correspondiente
                                if isinstance(ttp, str):
                                    ttp_counter += 1
                                    copy_file_to_path(item, save_path, ttp)
                            if ttp_counter == 0:
                                raise NotTTPatTags('.yaml sin TTP encontrada en los tags')
                    
                        else:
                            # En el caso de que la etiquete exista pero su contenido sea vacio (NoneType), accedemos a la etiqueta 'description' para buscar las TTP´s
                            has_label_without_content_tags.append(item)
                            if 'description' in document:
                                description = [document.get('description', [])]
                                # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                                for tag in range(len(description)):
                                    ttp = find_techniques_in_list(description[tag], techniques_list)
                                    if not isinstance(ttp, type(None)):
                                        if not os.path.exists(os.path.join(save_path, ttp)):
                                            os.makedirs(os.path.join(save_path, ttp))
                                    else:
                                        ttp='T0000'
                                    copy_file_to_path(item, save_path, ttp)
                    # Sigma HQ 3: Buscamos específicamente el método en la etiqueta 'mitreattack'
                    elif 'mitreattack' in document:
                        # Si encontramos el tag 'mitreattack' en el yaml lo guardamos como una variable 
                        mitreattacks = document.get('mitreattack', [])
                        # has_label_mitreattack.append(item)
                        # Si el resultado de esa variable no está vacío y contiene información lo convertimos en una lista
                        if not isinstance(mitreattacks, type(None)):
                            # has_label_with_content_mitreattack.append(item)
                            # Convertimos en lista el contenido de la etiqueta
                            mitreattacks = [mitreattacks]
                            if isinstance(mitreattacks, list):
                                # Chequeamos que se haya podido generar correctamente la lista y la vamos recorriendo añadiendo las ttps encontradas en el listado de Mitre
                                # Guardamos el fichero en la carpeta correspondiente.
                                for tag in range(len(mitreattacks)):
                                    ttp = find_techniques_in_list(mitreattacks[tag], techniques_list)
                                    if not isinstance(ttp, type(None)):
                                        if not os.path.exists(os.path.join(save_path, ttp)):
                                            os.makedirs(os.path.join(save_path, ttp))
                                    else:
                                        ttp = 'T0000'
                                    # shutil.copy(item,os.path.join(save_path, ttp))
                                    copy_file_to_path(item, save_path, ttp)
                                    print('Copiado correctamente a la carpeta de la TTP: '+ ttp)   
                    
                    else:
                        no_label_tags.append(item)
                        # Repetimos la busqueda de TTP en la etiqueta description en los casos en los que 'tags' no exista.
                        if 'description' in document:
                            description = [document.get('description', [])]
                            # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                            for tag in range(len(description)):
                                ttp = find_techniques_in_list(description[tag], techniques_list)
                                if not isinstance(ttp, type(None)):
                                    if not os.path.exists(os.path.join(save_path, ttp)):
                                        os.makedirs(os.path.join(save_path, ttp))
                                else:
                                    ttp='T0000'
                                copy_file_to_path(item, save_path, ttp)
                        else:
                            copy_file_to_path(item, save_path, 'T0000')
                # Excepción que controla errores de carga del fichero, moviendolo a T0000. 
                except yaml.YAMLError as e:
                    error_load.append(item)
                    ttp = find_techniques_in_list(item, techniques_list)
                    if not isinstance(ttp, type(None)):
                        print(f'Error al cargar el archivo YAML: {e}\nEncontrada TTP en la path del fichero')
                        error_load_ttp_in_path_name.append(item)
                        copy_file_to_path(item, save_path, ttp)
                    else:
                        print(f"Error al cargar el archivo YAML: {e}")
                        copy_file_to_path(item, save_path, 'T0000')
                # Con esta excepción llegamos si hemos recorrido la lista de tags y no hemos encontrado ninguna TTP, por lo que revisamos la etiqueta dando lugar a 3 casuísticas,
                # la primera que encuentre dicha etiqueta, en cuyo caso se revisa el contenido, si dentro de este se encuentra alguna TTP se mandara a la carpeta correspondiente. En caso de no haber coincidencia, al igual que ocurrirá si no hay etiqueta 'description' el fichero será copiado a T000.
                except NotTTPatTags as e:
                    print(f"{e}")
                    not_ttp_in_tags.append(item)
                    if 'description' in document:
                        description = [document.get('description', [])]
                        # Si no se encuentra ninguna, directamente guardamos en la carpeta T0000 y en caso contrario en la correspondiente a la TTP encontrada
                        for tag in range(len(description)):
                            ttp = find_techniques_in_list(description[tag],techniques_list)
                            if not isinstance(ttp, type(None)):
                                not_ttp_in_tags_with_ttp_in_description.append(item)
                                pass
                            else:
                                ttp='T0000'
                            copy_file_to_path(item, save_path, ttp)
                    else:
                        not_ttp_in_tags_without_description.append(item)
                        copy_file_to_path(item, save_path, 'T0000')

    # return num_rules, has_label_tags, has_label_with_content_tags, has_label_without_content_tags, no_label_tags, not_ttp_in_tags, not_ttp_in_tags_with_ttp_in_description, not_ttp_in_tags_without_description, error_load, error_load_ttp_in_path_name, ttps_in_tags_not_in_techniques, ttps_in_tags_not_in_techniques_items
    return num_rules

In [514]:
def get_resume_df(matrix_output_path, matrix):
    '''
    Función para generar el dataframe que resume los datos obtenidos
    '''
    resume_data = []
    for path in get_list_of_files_sub(matrix_output_path): 
        rule = os.path.basename(path)
        
        parts = path.split(os.sep)
        ttp = parts[-2]
        source = parts[-3]
        
        resume_data.append([rule, ttp, source])

    resume_df = pd.DataFrame(resume_data, columns=['rule', 'ttp', 'source'])
    resume_df['ttp'] = resume_df['ttp'].str.strip()
    resume_df['matrix'] = matrix
    ttp_resume_df = resume_df.groupby('ttp')['rule'].count().reset_index().sort_values(by='rule', ascending=False)
    no_t000_resume_df = resume_df[resume_df['ttp']!='T0000']
    no_t000_resume_df = no_t000_resume_df.reset_index(drop=True)
    return resume_df, ttp_resume_df, no_t000_resume_df

#### **Parámetros**

In [515]:
temp_path = str(Path(os.path.expanduser('~/Downloads')))
save_path = os.path.join(os.getcwd(), 'outputs')
matrix = 'enterprise' # enterprise / ics / mobile

In [516]:
src = get_data_from_branch(matrix)
techniques = get_list_techniques_from_stix2(src)
techniques[:3]

['T1055.011', 'T1053.005', 'T1205.002']

#### **1. UCM Catalog 2024**

**Importante**: debe ejecutarse previamente el código get_ttps_from_CP_files\get_CP_techniques.ipynb

No solo refactorizamos si no que en este caso vamos a modificar el código para guardar como .md

In [517]:
path_ucmcatalog2024 = os.path.join(os.path.join(os.path.dirname(os.getcwd()),'get_ttps_from_CP_files', 'outputs', matrix), f"[MITRE-{matrix}]_techniques_UCMCatalog2024_withquery.csv")
ucmcatalog2024_df = pd.read_csv(path_ucmcatalog2024, sep=';', quotechar='"')

##### **1.1 Sentinel**

In [518]:
# FIltramos el df original por las reglas pertenecientes a Sentinel
ucmcatalog2024_sentinel_df = ucmcatalog2024_df[ucmcatalog2024_df['source']=='Sentinel']
# Asignamos la carpeta principal de guardado
save_path_ucmcatalog2024_sentinel = os.path.join(save_path, matrix, 'UCM Catalog 2024 Sentinel')

In [519]:
# Ejecutamos la calsificación de reglas Sentinel
num_rules = sentineldf_to_md(ucmcatalog2024_sentinel_df, save_path_ucmcatalog2024_sentinel, techniques)

Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Sentinel\T1001\T1001 - CyberProof - Fortinet - High Severity Events.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Sentinel\T1003\T1003 - CyberProof - Azure Active Directory - Rare subscri.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Sentinel\T1003\T1003 - Powershell Empire Cmdlets Executed in Command Line.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Sentinel\T1003\T1003 - Credential Dumping Tools - File Artifacts.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_clas

In [520]:
try:
    print(f"TTP's UCM Catalog 2024 Sentinel - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

    print(f'Reglas totales para asignar: {len(num_rules)}')
    print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
    unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_ucmcatalog2024_sentinel))
    print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
    print("Resumen reglas asignadas por TTP: ")

    # get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_ucmcatalog2024_sentinel))
    # Mostramos las 5 primeras ttp's por número de reglas
    pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_ucmcatalog2024_sentinel)).items()), columns=['TTP', 'Reglas']).head(5)
except:
    print('No se ha generado ningun fichero')

TTP's UCM Catalog 2024 Sentinel - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 1957
Reglas únicas para asignar: 1634
Número de TTP's únicas identificadas excluyendo T0000: 176
Resumen reglas asignadas por TTP: 


##### **1.2 Splunk**

In [521]:
# Filtramos el df original por las reglas pertenecientes a Splunk
ucmcatalog2024_splunk_df = ucmcatalog2024_df[ucmcatalog2024_df['source']=='Splunk']
# Asignamos la carpeta principal de guardado
save_path_ucmcatalog2024_splunk = os.path.join(save_path, matrix, 'UCM Catalog 2024 Splunk')

In [522]:
# Ejecutamos la calsificación de reglas Splunk
num_rules = splunkdf_to_md(ucmcatalog2024_splunk_df, save_path_ucmcatalog2024_splunk, techniques)

Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Splunk\ T1070\ T1070 - CyberProof - Okta - Account Created and Deleted in.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Splunk\ T1070\ T1070 - CyberProof - WinEventLog - Windows Server Logs Cle.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Splunk\ T1078\ T1078 - CyberProof - Okta - Phishing Detection with FastPa.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Splunk\ T1078\ T1078 - CyberProof - Okta - Risk Threshold Exceeded.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_

In [523]:
try:
    print(f"TTP's UCM Catalog 2024 Splunk - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

    print(f'Reglas totales para asignar: {len(num_rules)}')
    print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
    unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_ucmcatalog2024_splunk))
    print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
    print("Resumen reglas asignadas por TTP: ")

    # get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_ucmcatalog2024_splunk))
    # Mostramos las 5 primeras ttp's por número de reglas
    pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_ucmcatalog2024_splunk)).items()), columns=['TTP', 'Reglas']).head(5)
except:
    print('No se ha generado ningun fichero')

TTP's UCM Catalog 2024 Splunk - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 105
Reglas únicas para asignar: 91
Número de TTP's únicas identificadas excluyendo T0000: 40
Resumen reglas asignadas por TTP: 


##### **1.3 Qradar**

In [524]:
# Filtramos el df original por las reglas pertenecientes a Qradar
ucmcatalog2024_qradar_df = ucmcatalog2024_df[ucmcatalog2024_df['source']=='Qradar']
# Asignamos la carpeta principal de guardado
save_path_ucmcatalog2024_qradar = os.path.join(save_path, matrix, 'UCM Catalog 2024 Qradar')

In [525]:
# Ejecutamos la calsificación de reglas Qradar
num_rules = qradardf_to_md(ucmcatalog2024_qradar_df, save_path_ucmcatalog2024_qradar, techniques)

Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Qradar\ T1003\ T1003 - [CyberProof] - [AD] - Directory Service Replicatio.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Qradar\ T1021\ T1021 - [CyberProof] - [Firewall] - Internal SMB Scan.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Qradar\ T1021\ T1021 - [CyberProof] - [Windows] - Suspicious RDP Redirect.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Qradar\ T1021\ T1021 - [CyberProof] - [Windows] - MMC Spawns Windows Shel.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rule

Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Qradar\ T1021\ T1021 - [AtomicRedTeam] - [Impacket #2] - WMIexec executio.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Qradar\ T1021\ T1021 - [CyberProof] - [Firewall] - Internal RDP Scan.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Qradar\ T1021.001\ T1021.001 - [CyberProof] - [Windows] - Suspicious RDP Redirect.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Qradar\ T1027\ T1027 - [AtomicRedTeam] - [Powershell #1] - PowerShell -en.md
Archivo guardado correctamente: c:\Users\jelopez\Documents\CyberProof\python\develop\

In [526]:
try:
    print(f"TTP's UCM Catalog 2024 Qradar - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

    print(f'Reglas totales para asignar: {len(num_rules)}')
    print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
    unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_ucmcatalog2024_qradar))
    print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
    print("Resumen reglas asignadas por TTP: ")

    # get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_ucmcatalog2024_qradar))
    # Mostramos las 5 primeras ttp's por número de reglas
    pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_ucmcatalog2024_qradar)).items()), columns=['TTP', 'Reglas']).head(5)
except:
    print('No se ha generado ningun fichero')

TTP's UCM Catalog 2024 Qradar - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 206
Reglas únicas para asignar: 201
Número de TTP's únicas identificadas excluyendo T0000: 90
Resumen reglas asignadas por TTP: 


#### **2. TTP's Mappings**

In [527]:
# Asignamos las path necesarias: url del zip, path de guardado
zip_mappings = 'https://github.com/center-for-threat-informed-defense/security-stack-mappings/archive/refs/heads/main.zip'
save_path_mappings = os.path.join(save_path, matrix, 'Mappings')
unzip_folder = r'security-stack-mappings-main/mappings/'

In [528]:
download_folder_unzip(zip_mappings, unzip_folder, temp_path)

Descarga y descompresión completadas correctamente.


In [529]:
# Creamos la carpeta de guardado
create_output_folder(save_path_mappings)

In [530]:
# Obtenemos el número de reglas a clasificar
files = get_list_of_files_sub(os.path.join(temp_path, unzip_folder))
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}") 

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 343


In [531]:
num_rules = yaml_classifier_techniques(files, techniques, save_path_mappings)
# Eliminamos la carpeta en la que descomprimimos los archivos
delete_folder(os.path.join(temp_path, 'security-stack-mappings-main'))

Regla asignada correctamente a la carpeta de la TTP: T1110.001
Regla asignada correctamente a la carpeta de la TTP: T1110.002
Regla asignada correctamente a la carpeta de la TTP: T1110.003
Regla asignada correctamente a la carpeta de la TTP: T1110.004
Regla asignada correctamente a la carpeta de la TTP: T1078.004
Regla asignada correctamente a la carpeta de la TTP: T1110
Regla asignada correctamente a la carpeta de la TTP: T1078
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T1110.001
Regla asignada correctamente a la carpeta de la TTP: T1498.001
Regla asignada correctamente a la carpeta de la TTP: T1491.002
Regla asignada correctamente a la carpeta de la TTP: T1568.002
Regla asignada correctamente a la carpeta de la TTP: T1071.004
Regla asignada correctamente a la carpeta de la TTP: T1552.005
Regla asignada correctamente a la carpeta de la TTP: T1565.001
Regla as

In [532]:
print(f"TTP's Mappings - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_mappings))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_mappings))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_mappings)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Mappings - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 126
Reglas únicas para asignar: 126
Número de TTP's únicas identificadas excluyendo T0000: 406
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T0000,126
1,T1078,39
2,T1078.004,33
3,T1110,32
4,T1190,29


#### **3. TTP's Azure-Sentinel Hunting Queries**

In [533]:
# Asignamos las path necesarias: url del zip, path de guardado
zip_sentinel_hq = 'https://github.com/Azure/Azure-Sentinel/archive/refs/heads/master.zip'
save_path_sentinel_hq = os.path.join(save_path, matrix, 'Azure Sentinel Hunting Queries')
unzip_folder = r'Azure-Sentinel-master/Hunting Queries/'

In [534]:
create_output_folder(save_path_sentinel_hq)

In [535]:
download_folder_unzip(zip_sentinel_hq, unzip_folder, temp_path)

Descarga y descompresión completadas correctamente.


In [536]:
# Obtenemos el número de reglas a clasificar
files = get_list_of_files_sub(os.path.join(temp_path, unzip_folder))
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}") 

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 1090


In [537]:
num_rules = yaml_classifier_techniques(files, techniques, save_path_sentinel_hq)
delete_folder(os.path.join(temp_path, 'Azure-Sentinel-master'))

Regla asignada correctamente a la carpeta de la TTP: T1567
Regla asignada correctamente a la carpeta de la TTP: T1102
Regla asignada correctamente a la carpeta de la TTP: T1204
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T1105
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T1071
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T1119
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T1114
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T1011
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T10

In [538]:
print(f"TTP's Azure Sentinel Hunting Queries - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_sentinel_hq))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sentinel_hq))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sentinel_hq)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Azure Sentinel Hunting Queries - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 1084
Reglas únicas para asignar: 1083
Número de TTP's únicas identificadas excluyendo T0000: 95
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T0000,1082
1,T1219,163
2,T1078,39
3,T1190,14
4,T1078.004,12


#### **4. TTP's Netevert [BlueTeamLabs/sentinel-attack]**

Nota: en el link del repositorio apunta a  https://github.com/BlueTeamLabs/sentinel-attack pero al acceder redirige a https://github.com/netevert/sentinel-attack

Para este caso el id de la técnica está disponible en el título del archivo por lo que simplifica nuestro árbol de búsqueda.

In [539]:
zip_sentinel_attack = 'https://github.com/netevert/sentinel-attack/archive/refs/heads/master.zip'
save_path_sentinel_attack = os.path.join(save_path, matrix, 'Netevert sentinel-attack')

In [540]:
create_output_folder(save_path_sentinel_attack)

In [541]:
download_unzip(zip_sentinel_attack, temp_path)

In [542]:
files = get_list_of_files_sub(temp_path+r'\sentinel-attack-master')
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}") 

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 174


In [543]:
num_rules = txt_basename_classifier_techniques(files, techniques, save_path_sentinel_attack)
delete_folder(os.path.join(temp_path, 'sentinel-attack-master'))

No identificada ttp T0000 en el archivo C:\Users\jelopez\Downloads\sentinel-attack-master\detections\T0000_Console_History.txt
No identificada ttp T0000 en el archivo C:\Users\jelopez\Downloads\sentinel-attack-master\detections\T0000_Named_Pipes.txt
No identificada ttp T0000 en el archivo C:\Users\jelopez\Downloads\sentinel-attack-master\detections\T0000_Named_Pipes_CobaltStrike.txt
No identificada ttp T0000 en el archivo C:\Users\jelopez\Downloads\sentinel-attack-master\detections\T0000_Remotely_Query_Login_Sessions_Network.txt
No identificada ttp T0000 en el archivo C:\Users\jelopez\Downloads\sentinel-attack-master\detections\T0000_Remotely_Query_Login_Sessions_Process.txt
No identificada ttp T0000 en el archivo C:\Users\jelopez\Downloads\sentinel-attack-master\detections\T0000_Suspicious_Filename_Used.txt
No identificada ttp T1002 en el archivo C:\Users\jelopez\Downloads\sentinel-attack-master\detections\T1002_Data_Compressed.txt
Identificada ttp T1003 en el archivo C:\Users\jelopez

In [544]:
print(f"TTP's Netevert [BlueTeamLabs/sentinel-attack] - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_sentinel_attack))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sentinel_attack))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sentinel_attack)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Netevert [BlueTeamLabs/sentinel-attack] - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 119
Reglas únicas para asignar: 119
Número de TTP's únicas identificadas excluyendo T0000: 36
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T0000,67
1,T1003,5
2,T1047,5
3,T1012,2
4,T1018,2


#### **5. TTP's Elastic - 1**

In [545]:
zip_elastic_1 = 'https://github.com/elastic/detection-rules/archive/refs/heads/main.zip'
unzip_folder = r'detection-rules-main/rules'
save_path_elastic_1 = os.path.join(save_path, matrix, 'Elastic 1')

In [546]:
create_output_folder(save_path_elastic_1)

In [547]:
download_folder_unzip(zip_elastic_1, 'detection-rules-main/rules', temp_path)

Descarga y descompresión completadas correctamente.


In [548]:
files = get_list_of_files_sub(os.path.join(temp_path, unzip_folder))
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}")

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 1131


In [549]:
num_rules = toml_classifier_techniques_elastic(files, techniques, save_path_elastic_1)

No se ha podido identificar ninguna ttp en: C:\Users\jelopez\Downloads\detection-rules-main/rules\apm\apm_403_response_to_a_post.toml
No se ha podido identificar ninguna ttp en: C:\Users\jelopez\Downloads\detection-rules-main/rules\apm\apm_405_response_method_not_allowed.toml
No se ha podido identificar ninguna ttp en: C:\Users\jelopez\Downloads\detection-rules-main/rules\apm\apm_sqlmap_user_agent.toml
Identificada/s ttp's ['T1105'] en: C:\Users\jelopez\Downloads\detection-rules-main/rules\cross-platform\command_and_control_google_drive_malicious_file_download.toml
Identificada/s ttp's ['T1571'] en: C:\Users\jelopez\Downloads\detection-rules-main/rules\cross-platform\command_and_control_non_standard_ssh_port.toml
Identificada/s ttp's ['T1539'] en: C:\Users\jelopez\Downloads\detection-rules-main/rules\cross-platform\credential_access_cookies_chromium_browsers_debugging.toml
Identificada/s ttp's ['T1036'] en: C:\Users\jelopez\Downloads\detection-rules-main/rules\cross-platform\defense_ev

In [550]:
delete_folder(os.path.join(temp_path, 'detection-rules-main'))

Directorio C:\Users\jelopez\Downloads\detection-rules-main eliminado correctamente.


In [551]:
print(f"TTP's Elastic 1 - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_elastic_1))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_elastic_1))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_elastic_1)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Elastic 1 - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 1129
Reglas únicas para asignar: 1127
Número de TTP's únicas identificadas excluyendo T0000: 314
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T1059,131
1,T1562,91
2,T0000,80
3,T1059.001,53
4,T1078,53


#### **6. TTP's Elastic - 2**

In [552]:
zip_elastic_2 = 'https://github.com/elastic/detection-rules/archive/refs/heads/main.zip'
unzip_folder = r'detection-rules-main/rules_building_block'
save_path_elastic_2 = os.path.join(save_path, matrix, 'Elastic 2')

In [553]:
create_output_folder(save_path_elastic_2)

In [554]:
download_folder_unzip(zip_elastic_2, 'detection-rules-main/rules_building_block', temp_path)

Descarga y descompresión completadas correctamente.


In [555]:
files = get_list_of_files_sub(os.path.join(temp_path, unzip_folder))
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}") 

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 111


In [556]:
num_rules = toml_classifier_techniques_elastic(files, techniques, save_path_elastic_2)

Identificada/s ttp's ['T1560'] en: C:\Users\jelopez\Downloads\detection-rules-main/rules_building_block\collection_archive_data_zip_imageload.toml
Identificada/s ttp's ['T1074', 'T1074.001', 'T1560', 'T1560.001', 'T1132', 'T1132.001', 'T1027'] en: C:\Users\jelopez\Downloads\detection-rules-main/rules_building_block\collection_common_compressed_archived_file.toml
Identificada/s ttp's ['T1074', 'T1074.001'] en: C:\Users\jelopez\Downloads\detection-rules-main/rules_building_block\collection_files_staged_in_recycle_bin_root.toml
Identificada/s ttp's ['T1115'] en: C:\Users\jelopez\Downloads\detection-rules-main/rules_building_block\collection_linux_suspicious_clipboard_activity.toml
Identificada/s ttp's ['T1114', 'T1114.001'] en: C:\Users\jelopez\Downloads\detection-rules-main/rules_building_block\collection_outlook_email_archive.toml
Identificada/s ttp's ['T1560', 'T1059', 'T1059.001'] en: C:\Users\jelopez\Downloads\detection-rules-main/rules_building_block\collection_posh_compression.toml

In [557]:
delete_folder(os.path.join(temp_path, 'detection-rules-main'))

Directorio C:\Users\jelopez\Downloads\detection-rules-main eliminado correctamente.


In [558]:
print(f"TTP's Elastic 2 - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_elastic_2))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_elastic_2))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_elastic_2)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Elastic 2 - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 110
Reglas únicas para asignar: 110
Número de TTP's únicas identificadas excluyendo T0000: 118
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T1036,14
1,T1082,9
2,T1036.001,8
3,T1059,8
4,T1218,7


#### **7. TTP's Atomic threat**

In [559]:
zip_atomic = 'https://github.com/krakow2600/atomic-threat-coverage/archive/refs/heads/master.zip'
save_path_atomic = os.path.join(save_path, matrix, 'Atomic')
unzip_folder = r'atomic-threat-coverage-master/Atomic_Threat_Coverage/'

In [560]:
download_folder_unzip(zip_atomic, unzip_folder, temp_path)

Descarga y descompresión completadas correctamente.


In [561]:
files = get_list_of_files_sub(os.path.join(temp_path, unzip_folder))
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}")

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 534


In [562]:
num_rules = md_classifier_techniques(files, techniques, save_path_atomic)

Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T00

In [563]:
delete_folder(os.path.join(temp_path, 'atomic-threat-coverage-master'))

Directorio C:\Users\jelopez\Downloads\atomic-threat-coverage-master eliminado correctamente.


In [564]:
print(f"TTP's Atomic - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_atomic))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_atomic))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_atomic)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Atomic - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 534
Reglas únicas para asignar: 534
Número de TTP's únicas identificadas excluyendo T0000: 77
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T0000,471
1,T1003,29
2,T1036,21
3,T1053,13
4,T1055,12


#### **8. TTP's Yara 1**

In [565]:
zip_yara_1 = 'https://github.com/Yara-Rules/rules/archive/refs/heads/master.zip'
save_path_yara_1 = os.path.join(save_path, matrix, 'Yara 1')
unzip_folder = r'rules-master/'

In [566]:
download_unzip(zip_yara_1, temp_path)

In [567]:
create_output_folder(save_path_yara_1)

In [568]:
files = get_list_of_files_sub(os.path.join(temp_path, unzip_folder))
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}")

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 582


In [569]:
num_rules = yara_classifier_techniques(files, techniques, save_path_yara_1)

Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T00

In [570]:
delete_folder(os.path.join(temp_path, 'rules-master'))

Directorio C:\Users\jelopez\Downloads\rules-master eliminado correctamente.


In [571]:
print(f"TTP's Yara 1 - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_yara_1))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_yara_1))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_yara_1)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Yara 1 - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 566
Reglas únicas para asignar: 566
Número de TTP's únicas identificadas excluyendo T0000: 0
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T0000,566


#### **9. TTP's Yara 2**

In [572]:
zip_yara_2 = 'https://github.com/bartblaze/Yara-rules/archive/refs/heads/master.zip'
save_path_yara_2 = os.path.join(save_path, matrix, 'Yara 2')
unzip_folder = r'Yara-rules-master/'

In [573]:
download_unzip(zip_yara_2, temp_path)

In [574]:
create_output_folder(save_path_yara_2)

In [575]:
files = get_list_of_files_sub(os.path.join(temp_path, unzip_folder))
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}") 

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 96


In [576]:
num_rules = yara_classifier_techniques(files, techniques, save_path_yara_2)

Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T00

In [577]:
delete_folder(os.path.join(temp_path, 'Yara-rules-master'))

Directorio C:\Users\jelopez\Downloads\Yara-rules-master eliminado correctamente.


In [578]:
print(f"TTP's Yara 2 - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_yara_2))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_yara_2))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_yara_2)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Yara 2 - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 91
Reglas únicas para asignar: 91
Número de TTP's únicas identificadas excluyendo T0000: 2
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T0000,91
1,T1021,1
2,T1021.005,1


### **10. TTP's Sigma HQ**

#### **10.1. TTP's Sigma HQ 1**

In [579]:
zip_sigma_hq1 = 'https://github.com/SigmaHQ/sigma/archive/refs/heads/master.zip'
save_path_sigma_hq1 = os.path.join(save_path, matrix, 'Sigma HQ 1')

In [580]:
create_output_folder(save_path_sigma_hq1)

In [581]:
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules', temp_path)
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules-compliance', temp_path)
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules-dfir', temp_path)
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules-emerging-threats', temp_path)
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules-placeholder', temp_path)
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules-threat-hunting', temp_path)
download_folder_unzip(zip_sigma_hq1, 'sigma-master/rules-placeholder', temp_path)

Descarga y descompresión completadas correctamente.
Descarga y descompresión completadas correctamente.
Descarga y descompresión completadas correctamente.
Descarga y descompresión completadas correctamente.
Descarga y descompresión completadas correctamente.
Descarga y descompresión completadas correctamente.
Descarga y descompresión completadas correctamente.


In [582]:
files = get_list_of_files_sub(temp_path+r'\sigma-master')
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}") 

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 3371


In [583]:
num_rules = yaml_classifier_techniques_sigma(files, techniques, save_path_sigma_hq1)

Regla asignada correctamente a la carpeta de la TTP: T1190
Regla asignada correctamente a la carpeta de la TTP: T1190
Regla asignada correctamente a la carpeta de la TTP: T1190
Regla asignada correctamente a la carpeta de la TTP: T1190
Regla asignada correctamente a la carpeta de la TTP: T1190
Regla asignada correctamente a la carpeta de la TTP: T1190
Regla asignada correctamente a la carpeta de la TTP: T1498
Regla asignada correctamente a la carpeta de la TTP: T1070
Regla asignada correctamente a la carpeta de la TTP: T1609
Regla asignada correctamente a la carpeta de la TTP: T1611
Regla asignada correctamente a la carpeta de la TTP: T1036.005
Regla asignada correctamente a la carpeta de la TTP: T1611
Regla asignada correctamente a la carpeta de la TTP: T1069.003
Regla asignada correctamente a la carpeta de la TTP: T1087.004
Regla asignada correctamente a la carpeta de la TTP: T1552.007
Regla asignada correctamente a la carpeta de la TTP: T1136
Regla asignada correctamente a la carpet

In [584]:
delete_folder(os.path.join(temp_path, 'sigma-master'))

Directorio C:\Users\jelopez\Downloads\sigma-master eliminado correctamente.


In [585]:
print(f"TTP's Sigma HQ 1 - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_sigma_hq1))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq1))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq1)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Sigma HQ 1 - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 3342
Reglas únicas para asignar: 3342
Número de TTP's únicas identificadas excluyendo T0000: 375
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T0000,452
1,T1059.001,201
2,T1218,137
3,T1190,119
4,T1562.001,101


#### **10.2. TTP's Sigma HQ 2**

In [586]:
zip_sigma_hq2 = 'https://github.com/mdecrevoisier/SIGMA-detection-rules/archive/refs/heads/main.zip'
save_path_sigma_hq2 = os.path.join(save_path, matrix, 'Sigma HQ 2')
unzip_folder = r'\SIGMA-detection-rules-main'

In [587]:
create_output_folder(save_path_sigma_hq2)

In [588]:
download_unzip(zip_sigma_hq2, temp_path)

In [589]:
files = get_list_of_files_sub(temp_path+unzip_folder)
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}")

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 351


In [590]:
num_rules = yaml_classifier_techniques_sigma(files, techniques, save_path_sigma_hq2)

Regla asignada correctamente a la carpeta de la TTP: T1484.002
Regla asignada correctamente a la carpeta de la TTP: T1114.003
Regla asignada correctamente a la carpeta de la TTP: T1114
Regla asignada correctamente a la carpeta de la TTP: T1566
Regla asignada correctamente a la carpeta de la TTP: T1114
Regla asignada correctamente a la carpeta de la TTP: T1566
Regla asignada correctamente a la carpeta de la TTP: T1562.006
Regla asignada correctamente a la carpeta de la TTP: T1546
Regla asignada correctamente a la carpeta de la TTP: T1110.001
Regla asignada correctamente a la carpeta de la TTP: T1110.003
Regla asignada correctamente a la carpeta de la TTP: T1136
Regla asignada correctamente a la carpeta de la TTP: T1098
Regla asignada correctamente a la carpeta de la TTP: T1136
Regla asignada correctamente a la carpeta de la TTP: T1068
Regla asignada correctamente a la carpeta de la TTP: T1222.001
Regla asignada correctamente a la carpeta de la TTP: T1098
Regla asignada correctamente a l

In [591]:
delete_folder(os.path.join(temp_path, 'SIGMA-detection-rules-main'))

Directorio C:\Users\jelopez\Downloads\SIGMA-detection-rules-main eliminado correctamente.


In [592]:
print(f"TTP's Sigma HQ 2 - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_sigma_hq2))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq1))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq2)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Sigma HQ 2 - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 348
Reglas únicas para asignar: 348
Número de TTP's únicas identificadas excluyendo T0000: 110
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T1098,45
1,T1543.003,19
2,T1021.002,12
3,T1003.001,11
4,T1562.001,11


#### **10.3. TTP's Sigma HQ 3**

In [593]:
zip_sigma_hq3 = 'https://github.com/joesecurity/sigma-rules/archive/refs/heads/master.zip'
save_path_sigma_hq3 = os.path.join(save_path, matrix, 'Sigma HQ 3')

In [594]:
create_output_folder(save_path_sigma_hq3)

In [595]:
download_unzip(zip_sigma_hq3, temp_path)

In [596]:
files = get_list_of_files_sub(temp_path+r'\sigma-rules-master')
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}")

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 121


In [597]:
save_path_sigma_hq3

'c:\\Users\\jelopez\\Documents\\CyberProof\\python\\develop\\get_rules_and_classify_by_ttp\\outputs\\enterprise\\Sigma HQ 3'

In [598]:
save_path_sigma_hq3

'c:\\Users\\jelopez\\Documents\\CyberProof\\python\\develop\\get_rules_and_classify_by_ttp\\outputs\\enterprise\\Sigma HQ 3'

In [599]:
#KKK

In [600]:
num_rules = yaml_classifier_techniques_sigma(files, techniques, save_path_sigma_hq3)

Regla asignada correctamente a la carpeta de la TTP: T1497
Copiado correctamente a la carpeta de la TTP: T1497


Regla asignada correctamente a la carpeta de la TTP: T1574.002
Copiado correctamente a la carpeta de la TTP: T1574.002
Regla asignada correctamente a la carpeta de la TTP: T1490
Copiado correctamente a la carpeta de la TTP: T1490
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T1218.010
Copiado correctamente a la carpeta de la TTP: T1218.010
Regla asignada correctamente a la carpeta de la TTP: T1218.010
Copiado correctamente a la carpeta de la TTP: T1218.010
Regla asignada correctamente a la carpeta de la TTP: T1055
Copiado correctamente a la carpeta de la TTP: T1055
Regla asignada correctamente a la carpeta de la TTP: T0000
Regla asignada correctamente a la carpeta de la TTP: T1497
Copiado correctamente a la carpeta de la TTP: T1497
Regla asignada correctamente a la carpeta de la TTP: T0000
Error al cargar

In [601]:
delete_folder(os.path.join(temp_path, 'sigma-rules-master'))

Directorio C:\Users\jelopez\Downloads\sigma-rules-master eliminado correctamente.


In [602]:
print(f"TTP's Sigma HQ 3 - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_sigma_hq3))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq1))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq3)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Sigma HQ 3 - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 118
Reglas únicas para asignar: 118
Número de TTP's únicas identificadas excluyendo T0000: 9
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T0000,21
1,T1055,3
2,T1218.010,2
3,T1497,2
4,T1053,1


#### **10.4. TTP's Sigma HQ 4**

In [603]:
zip_sigma_hq4 = 'https://github.com/SigmaHQ/sigma/archive/refs/heads/master.zip'
save_path_sigma_hq4 = os.path.join(save_path, matrix, 'Sigma HQ 4')
unzip_folder = r'sigma-master/rules/'

In [604]:
download_folder_unzip(zip_sigma_hq4, unzip_folder, temp_path)

Descarga y descompresión completadas correctamente.


In [605]:
files = get_list_of_files_sub(os.path.join(temp_path, unzip_folder))
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}") 

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 2908


In [606]:
create_output_folder(save_path_sigma_hq4)

In [607]:
num_rules = yaml_classifier_techniques_sigma(files, techniques, save_path_sigma_hq4)

Regla asignada correctamente a la carpeta de la TTP: T1190
Regla asignada correctamente a la carpeta de la TTP: T1190
Regla asignada correctamente a la carpeta de la TTP: T1190
Regla asignada correctamente a la carpeta de la TTP: T1190
Regla asignada correctamente a la carpeta de la TTP: T1190
Regla asignada correctamente a la carpeta de la TTP: T1190
Regla asignada correctamente a la carpeta de la TTP: T1498
Regla asignada correctamente a la carpeta de la TTP: T1070
Regla asignada correctamente a la carpeta de la TTP: T1609
Regla asignada correctamente a la carpeta de la TTP: T1611
Regla asignada correctamente a la carpeta de la TTP: T1036.005
Regla asignada correctamente a la carpeta de la TTP: T1611
Regla asignada correctamente a la carpeta de la TTP: T1069.003
Regla asignada correctamente a la carpeta de la TTP: T1087.004
Regla asignada correctamente a la carpeta de la TTP: T1552.007
Regla asignada correctamente a la carpeta de la TTP: T1136
Regla asignada correctamente a la carpet

In [608]:
delete_folder(os.path.join(temp_path, 'sigma-master'))

Directorio C:\Users\jelopez\Downloads\sigma-master eliminado correctamente.


In [609]:
print(f"TTP's Sigma HQ 4 - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_sigma_hq4))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq1))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq4)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Sigma HQ 4 - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 2907
Reglas únicas para asignar: 2907
Número de TTP's únicas identificadas excluyendo T0000: 371
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T0000,341
1,T1059.001,173
2,T1218,126
3,T1562.001,100
4,T1027,87


#### **10.5. TTP's Sigma HQ 5**

In [610]:
zip_sigma_hq5 = 'https://github.com/socprime/socprime_sigma/archive/refs/heads/master.zip'
save_path_sigma_hq5 = os.path.join(save_path, matrix, 'Sigma HQ 5')
unzip_folder = r'socprime_sigma-master'

In [611]:
download_unzip(zip_sigma_hq5, temp_path)

In [612]:
files = get_list_of_files_sub(os.path.join(temp_path, unzip_folder))
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}") 

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 37


In [613]:
create_output_folder(save_path_sigma_hq5)

In [614]:
num_rules = yaml_classifier_techniques_sigma(files, techniques, save_path_sigma_hq5)

Error al cargar el archivo YAML: expected '<document start>', but found '<scalar>'
  in "C:\Users\jelopez\Downloads\socprime_sigma-master\Fancy_Bear\Rule\Fancy_Bear.yml", line 58, column 1
Regla asignada correctamente a la carpeta de la TTP: T0000
Error al cargar el archivo YAML: expected a single document in the stream
  in "C:\Users\jelopez\Downloads\socprime_sigma-master\FlawedAmmyy\Rule\FlawedAmmyy_RAT.yml", line 1, column 1
but found another document
  in "C:\Users\jelopez\Downloads\socprime_sigma-master\FlawedAmmyy\Rule\FlawedAmmyy_RAT.yml", line 21, column 1
Regla asignada correctamente a la carpeta de la TTP: T0000
.yaml sin TTP encontrada en los tags
Regla asignada correctamente a la carpeta de la TTP: T0000
Error al cargar el archivo YAML: while scanning for the next token
found character '\t' that cannot start any token
  in "C:\Users\jelopez\Downloads\socprime_sigma-master\InvisiMole\Rule\InvisiMole.yml", line 35, column 1
Regla asignada correctamente a la carpeta de la TTP

In [615]:
delete_folder(os.path.join(temp_path, unzip_folder))

Directorio C:\Users\jelopez\Downloads\socprime_sigma-master eliminado correctamente.


In [616]:
print(f"TTP's Sigma HQ 5 - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_sigma_hq5))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq10))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq5)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Sigma HQ 5 - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 12
Reglas únicas para asignar: 12
Número de TTP's únicas identificadas excluyendo T0000: 0
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T0000,12


#### **10.6. TTP's Sigma HQ 6**

No aplica, la url a la que apunta el repositorio está caída: https://www.signalscorps.com/blog/2021/sigma-rules-101-yaml

#### **10.7. TTP's Sigma HQ 7**

**Importante**: En este caso el repositorio almacena las reglas por ttp por lo que simplemente debemos mover los ficheros, pero incluimos un filtro para mover a la T0000 aquellas técnicas que no pertenezcan a la matriz en ejecución.

**Importante**: En origen vienen dos ficheros para cada regla .md y .yml. Nos quedamos unicamente con el .md ya que duplica la información y en este caso el .md parece ser mas completo que el .yml

In [617]:
zip_sigma_hq7 = 'https://github.com/P4T12ICK/Sigma-Rule-Repository/archive/refs/heads/master.zip'
save_path_sigma_hq7 = os.path.join(save_path, matrix, 'Sigma HQ 7')
unzip_folder = r'Sigma-Rule-Repository-master/detection-rules/'

In [618]:
download_folder_unzip(zip_sigma_hq7, unzip_folder, temp_path)

Descarga y descompresión completadas correctamente.


In [619]:
num_rules = sigma7_classifier_techniques(os.path.join(temp_path, unzip_folder),save_path_sigma_hq7, techniques)

Proceso finalizado!


In [620]:
delete_folder(os.path.join(temp_path, 'Sigma-Rule-Repository-master'))

Directorio C:\Users\jelopez\Downloads\Sigma-Rule-Repository-master eliminado correctamente.


In [621]:
print(f"TTP's Sigma HQ 7 - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_sigma_hq7))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq7))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq7)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Sigma HQ 7 - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 12
Reglas únicas para asignar: 12
Número de TTP's únicas identificadas excluyendo T0000: 8
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T0000,4
1,T1003,1
2,T1053,1
3,T1070,1
4,T1078,1


#### **10.8. TTP's Sigma HQ 8**

In [622]:
zip_sigma_hq8 = 'https://github.com/blacklanternsecurity/sigma-rules/archive/refs/heads/main.zip'
save_path_sigma_hq8 = os.path.join(save_path, matrix, 'Sigma HQ 8')
unzip_folder = r'sigma-rules-main/'

In [623]:
download_unzip(zip_sigma_hq8, temp_path)

In [624]:
files = get_list_of_files_sub(os.path.join(temp_path, unzip_folder))
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}") 

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 11


In [625]:
create_output_folder(save_path_sigma_hq8)

In [626]:
num_rules = yaml_classifier_techniques_sigma(files, techniques, save_path_sigma_hq8)

Regla asignada correctamente a la carpeta de la TTP: T1068
Regla asignada correctamente a la carpeta de la TTP: T1207
Regla asignada correctamente a la carpeta de la TTP: T1003.006
Regla asignada correctamente a la carpeta de la TTP: T1558.003
Regla asignada correctamente a la carpeta de la TTP: T1018
Regla asignada correctamente a la carpeta de la TTP: T1069.002
Regla asignada correctamente a la carpeta de la TTP: T1069.002
Regla asignada correctamente a la carpeta de la TTP: T1087.002
Regla asignada correctamente a la carpeta de la TTP: T1482
Error al cargar el archivo YAML: while scanning an alias
  in "C:\Users\jelopez\Downloads\sigma-rules-main/TA0008 - Lateral Movement\T1550 - Use Alternate Authentication Material\002 - Pass the Hash\ad_cs_relay.yml", line 18, column 25
expected alphabetic or numeric character, but found '\n'
  in "C:\Users\jelopez\Downloads\sigma-rules-main/TA0008 - Lateral Movement\T1550 - Use Alternate Authentication Material\002 - Pass the Hash\ad_cs_relay.ym

In [627]:
delete_folder(os.path.join(temp_path, unzip_folder))

Directorio C:\Users\jelopez\Downloads\sigma-rules-main/ eliminado correctamente.


In [628]:
print(f"TTP's Sigma HQ 8 - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_sigma_hq8))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq8))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq8)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Sigma HQ 8 - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 10
Reglas únicas para asignar: 10
Número de TTP's únicas identificadas excluyendo T0000: 9
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T1069.002,2
1,T1003.006,1
2,T1018,1
3,T1068,1
4,T1087.002,1


#### **10.9. TTP's Sigma HQ 9**

In [629]:
zip_sigma_hq9 = 'https://github.com/mdecrevoisier/SIGMA-detection-rules/archive/refs/heads/main.zip'
save_path_sigma_hq9 = os.path.join(save_path, matrix, 'Sigma HQ 9')
unzip_folder = r'SIGMA-detection-rules-main/'

In [630]:
download_unzip(zip_sigma_hq9, temp_path)

In [631]:
create_output_folder(save_path_sigma_hq9)

In [632]:
files = get_list_of_files_sub(os.path.join(temp_path, unzip_folder))
print(f"Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: {len(files)}") 

Número de archivos a clasificar* incluye reglas y otros posibles tipos de archivos: 351


In [633]:
num_rules = yaml_classifier_techniques_sigma(files, techniques, save_path_sigma_hq9)

Regla asignada correctamente a la carpeta de la TTP: T1484.002
Regla asignada correctamente a la carpeta de la TTP: T1114.003
Regla asignada correctamente a la carpeta de la TTP: T1114
Regla asignada correctamente a la carpeta de la TTP: T1566
Regla asignada correctamente a la carpeta de la TTP: T1114
Regla asignada correctamente a la carpeta de la TTP: T1566
Regla asignada correctamente a la carpeta de la TTP: T1562.006
Regla asignada correctamente a la carpeta de la TTP: T1546
Regla asignada correctamente a la carpeta de la TTP: T1110.001
Regla asignada correctamente a la carpeta de la TTP: T1110.003
Regla asignada correctamente a la carpeta de la TTP: T1136
Regla asignada correctamente a la carpeta de la TTP: T1098
Regla asignada correctamente a la carpeta de la TTP: T1136
Regla asignada correctamente a la carpeta de la TTP: T1068
Regla asignada correctamente a la carpeta de la TTP: T1222.001
Regla asignada correctamente a la carpeta de la TTP: T1098
Regla asignada correctamente a l

Regla asignada correctamente a la carpeta de la TTP: T1564
Regla asignada correctamente a la carpeta de la TTP: T1003
Regla asignada correctamente a la carpeta de la TTP: T1484.001
Regla asignada correctamente a la carpeta de la TTP: T1484.001
Regla asignada correctamente a la carpeta de la TTP: T1069.002
Regla asignada correctamente a la carpeta de la TTP: T1087.002
Regla asignada correctamente a la carpeta de la TTP: T1098
Regla asignada correctamente a la carpeta de la TTP: T1087
Regla asignada correctamente a la carpeta de la TTP: T1021
Regla asignada correctamente a la carpeta de la TTP: T1003.003
Regla asignada correctamente a la carpeta de la TTP: T1003.003
Regla asignada correctamente a la carpeta de la TTP: T1558.003
Regla asignada correctamente a la carpeta de la TTP: T1110
Regla asignada correctamente a la carpeta de la TTP: T1558.004
Regla asignada correctamente a la carpeta de la TTP: T1558
Regla asignada correctamente a la carpeta de la TTP: T1098
Regla asignada correctam

In [634]:
delete_folder(os.path.join(temp_path, unzip_folder))

Directorio C:\Users\jelopez\Downloads\SIGMA-detection-rules-main/ eliminado correctamente.


In [635]:
print(f"TTP's Sigma HQ 9 - Resumen clasificación de reglas de detección para la matriz {matrix.upper()}")

print(f'Reglas totales para asignar: {len(num_rules)}')
print(f'Reglas únicas para asignar: {len(get_unique_items(num_rules))}')
unique_ttps = get_unique_ttps_generated(get_list_of_files_sub(save_path_sigma_hq9))
print(f"Número de TTP's únicas identificadas excluyendo T0000: {len([ttp for ttp in unique_ttps if ttp != 'T0000'])}")
print("Resumen reglas asignadas por TTP: ")

# get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq9))
# Mostramos las 5 primeras ttp's por número de reglas
pd.DataFrame(list(get_dict_ttps_rules_asigned(get_list_of_files_sub(save_path_sigma_hq9)).items()), columns=['TTP', 'Reglas']).head(5)

TTP's Sigma HQ 9 - Resumen clasificación de reglas de detección para la matriz ENTERPRISE
Reglas totales para asignar: 348
Reglas únicas para asignar: 348
Número de TTP's únicas identificadas excluyendo T0000: 110
Resumen reglas asignadas por TTP: 


,TTP,Reglas
0,T1098,45
1,T1543.003,19
2,T1021.002,12
3,T1003.001,11
4,T1562.001,11


#### **11. Resumen ejecuciones**

In [636]:
output_path = os.path.join(save_path, matrix)

In [637]:
resume_df, ttp_resume_df, no_t000_resume_df = get_resume_df(output_path, matrix)

In [638]:
print(f'Datos básicos obtenidos de la ejecución relaizada:')
print(f'Se han clasificado un total de {resume_df.shape[0]} reglas para la matriz {matrix.upper()}')
print(f'{no_t000_resume_df.shape[0]} se han asignado a alguna TTP de la matriz {matrix.upper()}')

Datos básicos obtenidos de la ejecución relaizada:
Se han clasificado un total de 18478 reglas para la matriz ENTERPRISE
15152 se han asignado a alguna TTP de la matriz ENTERPRISE


In [639]:
# Guardado de las tablas completas
resume_df.to_csv(os.path.join(output_path,f'{matrix}-all_classified_rules.csv'), sep=';',encoding='utf8',index=False, quoting=csv.QUOTE_ALL)
ttp_resume_df.to_csv(os.path.join(output_path,f'{matrix}-ttp_all_classified_rules.csv'), sep=';',encoding='utf8',index=False, quoting=csv.QUOTE_ALL)
no_t000_resume_df.to_csv(os.path.join(output_path,f'{matrix}-classified_rules.csv'), sep=';',encoding='utf8',index=False, quoting=csv.QUOTE_ALL)

In [640]:
no_t000_resume_df

,rule,ttp,source,matrix
0,av_password_dumper.md,T1003,Atomic,enterprise
1,MP_0001_windows_asr_block_credential_stealing_...,T1003,Atomic,enterprise
2,sysmon_ghostpack_safetykatz.md,T1003,Atomic,enterprise
3,sysmon_lsass_memdump.md,T1003,Atomic,enterprise
4,sysmon_mimikatz_detection_lsass.md,T1003,Atomic,enterprise
...,...,...,...,...
15147,T1621 - CyberProof - Cloudtrail - AWS Multiple...,T1621,UCM Catalog 2024 Splunk,enterprise
15148,T1621 - CyberProof - Okta - MFA Bombing.md,T1621,UCM Catalog 2024 Splunk,enterprise
15149,T1621 - CyberProof - Okta - Mismatch for Verif...,T1621,UCM Catalog 2024 Splunk,enterprise
15150,HiddenVNC.yar,T1021,Yara 2,enterprise
